In [ ]:
import json

# Load previously saved parsed questions
with open("parsed_questions.json", "r", encoding="utf-8") as f:
    parsed_questions = json.load(f)

print(f"Total questions loaded: {len(parsed_questions)}")

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import numpy as np

# Load questions
questions = [q['question_text'] for q in parsed_questions]

# --- TF-IDF ---
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_embeddings = tfidf_vectorizer.fit_transform(questions)

# --- BERT embeddings ---
bert_model = SentenceTransformer('all-MiniLM-L6-v2')
bert_embeddings = bert_model.encode(questions)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Simulate user queries (can be real queries)
queries = [
    "Explain convergence in probability",
    "Define uniform continuity",
    "What is mean square convergence?"
]

def recommend_tfidf(query, top_k=3):
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return [questions[i] for i in top_idx], [sims[i] for i in top_idx]

def recommend_bert(query, top_k=3):
    q_vec = bert_model.encode([query])
    sims = cosine_similarity(q_vec, bert_embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return [questions[i] for i in top_idx], [sims[i] for i in top_idx]

# Example
for q in queries:
    recs_tfidf, scores = recommend_tfidf(q)
    print(f"\nQuery (TF-IDF): {q}")
    for i, (r, s) in enumerate(zip(recs_tfidf, scores), 1):
        print(f"{i}. {r[:200]}... (score={s:.3f})")

    recs_bert, scores = recommend_bert(q)
    print(f"\nQuery (BERT): {q}")
    for i, (r, s) in enumerate(zip(recs_bert, scores), 1):
        print(f"{i}. {r[:200]}... (score={s:.3f})")

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

# Simulate user queries (can be real queries)
queries = [
    "Explain convergence in probability",
    "Define uniform continuity",
    "What is mean square convergence?"
]

def recommend_tfidf(query, top_k=3):
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return [questions[i] for i in top_idx], [sims[i] for i in top_idx]

def recommend_bert(query, top_k=3):
    q_vec = bert_model.encode([query])
    sims = cosine_similarity(q_vec, bert_embeddings)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return [questions[i] for i in top_idx], [sims[i] for i in top_idx]

# Example
for q in queries:
    recs_tfidf, scores = recommend_tfidf(q)
    print(f"\nQuery (TF-IDF): {q}")
    for i, (r, s) in enumerate(zip(recs_tfidf, scores), 1):
        print(f"{i}. {r[:200]}... (score={s:.3f})")

    recs_bert, scores = recommend_bert(q)
    print(f"\nQuery (BERT): {q}")
    for i, (r, s) in enumerate(zip(recs_bert, scores), 1):
        print(f"{i}. {r[:200]}... (score={s:.3f})")


Query (TF-IDF): Explain convergence in probability
1. Does X n converges in Lp to 0 as n tends to infinity? Justify: 2 . Does Xn converges in probability to 0 as n tends to infinity? Justify:... (score=0.258)
2. Suppose that ECIXI) O _ then E(X) = {v(0). 2 . Suppose that E(lXI?) 0, then E(X2) = #p"(0).... (score=0.000)
3. : Let X be a random variable whose characteristic function is p.... (score=0.000)

Query (BERT): Explain convergence in probability
1. Does X n converges in Lp to 0 as n tends to infinity? Justify: 2 . Does Xn converges in probability to 0 as n tends to infinity? Justify:... (score=0.640)
2. Does Xn converges almost surely to 0 as n tends to infinity? Justify:... (score=0.610)
3. : Let X be a random variable whose characteristic function is p.... (score=0.210)

Query (TF-IDF): Define uniform continuity
1. Suppose that ECIXI) O _ then E(X) = {v(0). 2 . Suppose that E(lXI?) 0, then E(X2) = #p"(0).... (score=0.000)
2. : Let X be a random variable whose characteristic fu

In [ ]:
# Compute Accuracy
# Suppose you have ground truth indices of correct question in `questions`
ground_truth = [0, 2, 1]  # index of correct question for each query

correct = 0
for i, q in enumerate(queries):
    recs, _ = recommend_bert(q, top_k=1)  # top-1
    if recs[0] == questions[ground_truth[i]]:
        correct += 1

accuracy = correct / len(queries)
print(f"\nRecommendation Accuracy (Top-1, BERT): {accuracy:.2f}")


Recommendation Accuracy (Top-1, BERT): 0.67
